In [ ]:
from fmsae_repro import MODELS, RESULTS, LEGACY_ANALYSIS, load_manifest
import ezslide
import lazyslide as zs
import mesoslide as ms
import pandas as pd

In [ ]:
manifest = load_manifest()
slides = ezslide.open_slides(manifest, attach_images=False)


In [ ]:
from mesoslide.tools import LocalityConstrainedCoding

llc = LocalityConstrainedCoding(n_codewords=128, 
                                n_neighbors=16,
                                kmeans_backend='cuml',
                                random_state=0)
llc.fit(slides, 
        obsm_key="UNI_embedding", 
        tile_key="tiles", 
        device='cuda:0',
        verbose=100)

In [ ]:
for _, slide in slides.items():
    ms.tl.feature_extraction(
        slide,
        model="uni",
        tile_key="tiles",
        key_added="UNI_embedding",
        sparse=True,
        sparse_transform=llc.transform,
        sparse_key_added="UNI_SAE_v2",
        save=False,
    )

In [ ]:
from mesoslide.tools import FeatureSelector

selector = FeatureSelector(
    pct_threshold=0.002,
    max_score_threshold=0.1,
    n_chunks=40
)
selector.compute_activation_stats(slides, 
                                  feature_prefix='UNI_SAE_v2', 
                                  num_features=128)
g = selector.plot_feature_selection()


In [ ]:
highly_active_features_idx = selector.get_selected_indices()
len(highly_active_features_idx)

In [ ]:
# Cluster
from mesoslide.tools import FeatureClusterer

clusterer = FeatureClusterer(high_activation_threshold=0.5)
clusterer.compute_iou(slides, feature_prefix='UNI_SAE_v2',
                      feature_indices=highly_active_features_idx)
clusterer.cluster(threshold=25, criterion='maxclust')

In [ ]:
g = clusterer.plot_heatmap()


In [ ]:
exemplar_patches = ms.select_exemplar_patches(slides, [f'UNI_SAE_v2_{idx}' for idx in highly_active_features_idx])
exemplar_patches

In [ ]:
from mesoslide.plotting._patch_gallery import plot_feature_gallery
from mesoslide.preprocessing._extract_patches import extract_patches
import distinctipy
from matplotlib.colors import ListedColormap

# ── configure ──────────────────────────────────────────────────────────────────
groups_to_plot = list(range(1, 26))
colors = distinctipy.get_colors(25, rng=43)
cmap = ListedColormap(colors)

# ── extract pixel patches once, keyed by global feature index ──────────────────
exemplar_images = extract_patches(exemplar_patches, slides, channel_first=False, progress_bar=True)
feature_to_image = {
    int(name.rsplit('_', 1)[-1]): img
    for name, img in zip(exemplar_patches.obs['_feature_name'], exemplar_images)
}

# ── filter by group ────────────────────────────────────────────────────────────
ordered_idx      = clusterer.feature_indices_[clusterer.reordered_idx_]
reordered_groups = clusterer.reordered_clusters_.tolist()

mask      = [i for i, g in enumerate(reordered_groups) if g in groups_to_plot]
patches   = [feature_to_image[ordered_idx[i]] for i in mask]
group_ids = [reordered_groups[i] for i in mask]
labels    = [str(ordered_idx[i]) for i in mask]  # or [str(ordered_idx[i]) for i in mask] if you want labels

# ── plot ───────────────────────────────────────────────────────────────────────
fig, axs = plot_feature_gallery(
    images=patches,
    group_ids=group_ids,
    labels=labels,
    n_cols=10,
    patch_size=1.0,
    border_extend=0.05,
    border_alpha=1.0,
    cmap=cmap,
    fontsize=6,
)
# fig.savefig(f'{LEGACY_ANALYSIS}/SAE_paper/figures/Fig_3/exemplar_patches_cellular_features.png', dpi=300)

In [ ]:
patches = ms.select_top_patches(slides, 'UNI_SAE_v2_93', 20, take_every=100)
ms.plotting.plot_patch_gallery(patches, slides, return_fig=True)

In [ ]:
_ = ms.plotting.plot_feature_map(slides['LSP24530'], 'UNI_SAE_v2_93', 
                                 figsize=(10, 10), fill_alpha=1.0)

In [ ]:
from mesoslide.plotting import create_feature_report
ms.plotting._feature_reports.create_feature_report(
    slides=slides,
    feature_name='UNI_SAE_v2_106',
    fill_alpha=1.0,
    top_fraction=0.01,
    nrows=2,
    show_titles=True,
    output_path=f'{RESULTS}/figures/LLC_feature_reports/UNI_LLC_106.pdf',
)

In [ ]:
import joblib

joblib.dump(llc, f'{MODELS}/LLC/llc_model.pkl')